# OCR: V1 

V1 run Chandra2 to OCR a list of pdfs files contained in the input folder and saves the aggregated md of all pages and the OCR statistics (nb of tokens, characters and potential errors) into the output folder.

In [1]:
import os
from dotenv import load_dotenv
_ = load_dotenv(override=True)

In [2]:
# Pointe le client chandra vers notre serveur vLLM (nom DNS sur scirex-net).
# Doit être fait AVANT d'importer chandra.model.vllm car settings est chargé à l'import.
os.environ["VLLM_API_BASE"] = "http://chandra-vllm:8000/v1"
os.environ["VLLM_MODEL_NAME"] = "chandra"

In [4]:
from PIL import Image
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Any
import duckdb
from pdf2image import convert_from_path
import os


from chandra.model.vllm import generate_vllm
from chandra.model.schema import BatchInputItem
from chandra.output import parse_markdown, parse_chunks
from IPython.display import Markdown, display

import time
import pymupdf
from PIL import Image
import io

In [4]:
def pdf_to_images(pdf_path, max_pages = None, dpi = 192):
    '''Take a pdf, return a list of pages converted in png images'''
    # Open the document and count the number of pages
    doc = pymupdf.open(pdf_path)

    if max_pages is None:
        max_pages = doc.page_count
    else:
        max_pages = min(max_pages, doc.page_count)

    img_list = []
    for page in range(max_pages):
        pix = doc[page].get_pixmap(dpi=dpi)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        img_list.append(img)
    doc.close()
    
    return img_list


In [ ]:
def images_to_markdown(images, max_output_tokens=8192, max_workers=8):
    '''Receives a list of PIL images, returns the concatenated markdown and per-page stats.'''
    
    batch = [BatchInputItem(image=img, prompt_type="ocr_layout") for img in images]
    results = generate_vllm(
        batch,
        max_output_tokens=max_output_tokens,
        max_workers=max_workers,
    )

    per_page_md = []
    per_page_stats = []

    for i, result in enumerate(results):
        md = parse_markdown(result.raw)
        per_page_md.append(md)
        per_page_stats.append({
            'page': i,
            'tokens': result.token_count,
            'n_chars': len(md),
            'error': result.error,
        })
    
    full = ""
    for i, md in enumerate(per_page_md):
        full += f"\n\n<!-- ===== Page {i+1} ===== -->\n\n"
        full += md
        full += "\n"

    return full, per_page_stats

In [ ]:
def stats_path_for(pdf_path, output_dir):
    '''Compute the csv output path for storing the stats of each pdf'''
    return Path(output_dir) / "stats" / (Path(pdf_path).stem + ".csv")

In [ ]:
def save_stats(stats, pdf_path, output_dir):
    '''Save the per-page stats to a .csv file in output_dir, naming the file after the source PDF.'''
    output_csv = stats_path_for(pdf_path, output_dir)
    output_csv.parent.mkdir(parents = True, exist_ok=True)

    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["page", "tokens", "n_chars", 'n_images", "error"])
        writer.writeheader()
        writer.writerows(stats)
    
    return output_csv

In [ ]:
def output_path_for(pdf_path, output_dir):
    '''Compute the markdown output path for a given pdf'''
    return Path(output_dir) / (Path(pdf_path).stem + ".md")

In [ ]:
def save_markdown(md, pdf_path, output_dir):
    '''Save the markdown to output_dir, naming the file after the source PDF.'''
    output_md = output_path_for(pdf_path, output_dir)
    output_md.parent.mkdir(parents=True, exist_ok=True)
    
    header = f"<!-- Source: {Path(pdf_path).name} -->\n"
    output_md.write_text(header + md)
    
    return output_md

In [ ]:
pages = pdf_to_images("data/raw/pdfs/2401.01623.pdf")
md, stats = images_to_markdown(pages)
saved_path = save_markdown(md, "data/raw/pdfs/2401.01623.pdf", "data/ocr_hf/")
print(f"Wrote {saved_path}")

Wrote data/ocr_hf/2401.01623.md
